In [1]:
from csrio_image2biomass.configs.settings import AUGUMENTED_DATA_DIR
import polars as pl
train = pl.read_csv(AUGUMENTED_DATA_DIR / "train.csv")
val = pl.read_csv(AUGUMENTED_DATA_DIR / "val.csv")
test = pl.read_csv(AUGUMENTED_DATA_DIR / "test.csv")
train.sort("image_path").head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_hvflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1011485656_vflip.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
scaler = StandardScaler()
transformer = ColumnTransformer(
    transformers=[
        ("scaler", scaler, [
            "Dry_Clover_g",
            "Dry_Dead_g",
            "Dry_Green_g",
            "Dry_Total_g",
            "GDM_g"
        ])
    ], remainder="passthrough"
)
train_scaled = transformer.fit_transform(train.to_pandas())
val_scaled = transformer.transform(val.to_pandas())
test_scaled = transformer.transform(test.to_pandas())
columns = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g", "image_path"]
train_scaled_df = pl.DataFrame(train_scaled, schema=columns)
val_scaled_df = pl.DataFrame(val_scaled, schema=columns)
test_scaled_df = pl.DataFrame(test_scaled, schema=columns)
train_scaled_df

Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,image_path
object,object,object,object,object,str
-0.055749373358736344,-0.4961056502133328,-0.12717626318304592,-0.35262920626858196,-0.1549704011594446,"""train/ID875119737.jpg"""
0.34736355061822394,-0.6524143337270202,-0.9718035633374121,-1.003811738996064,-0.8130620694208961,"""train/ID249042826.jpg"""
5.415702609117339,0.3132760098299819,-1.0082961989319321,1.5386801350551238,1.5824059279896017,"""train/ID1831254380.jpg"""
-0.33766971676865587,-0.8158024651810342,-0.3587264959781683,-0.8182256871614001,-0.523706263678923,"""train/ID135365668.jpg"""
-0.5366723121168344,-0.8130958564966153,-0.9481224240807476,-1.4284392568163817,-1.2134242135217879,"""train/ID956512130.jpg"""
…,…,…,…,…,…
2.1697629846183935,-0.9675269826346248,-1.0428469811074055,-0.4288177213237703,-0.010129353151966053,"""train/ID1761544403_vflip.jpg"""
2.1697629846183935,-0.9675269826346248,-1.0428469811074055,-0.4288177213237703,-0.010129353151966053,"""train/ID1761544403_hvflip.jpg"""
-0.035019936343301084,-0.9675269826346248,-1.0428469811074055,-1.3667124506374095,-1.0681852505472555,"""train/ID1025234388_hflip.jpg"""


In [3]:
from csrio_image2biomass.utils.dataset import BiomassDataset
from csrio_image2biomass.utils.metric import weighted_r2_score, WeightedMSELoss
from torch.utils.data import DataLoader

train_dataset = BiomassDataset(train, img_dir=AUGUMENTED_DATA_DIR)
val_dataset = BiomassDataset(val, img_dir=AUGUMENTED_DATA_DIR)
test_dataset = BiomassDataset(test, img_dir=AUGUMENTED_DATA_DIR)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False, num_workers=16)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=16)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=16)

for images, targets in train_loader:
    print(images.shape)
    print(targets.shape)
    break

torch.Size([32, 3, 448, 224])
torch.Size([32, 5])


In [4]:
import torch
import torchvision.models as models
import torch.nn as nn

class BiomassModel(nn.Module):
    def __init__(self):
        super(BiomassModel, self).__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.fc = nn.Linear(self.backbone.fc.out_features, 5)

    def forward(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return torch.relu(x)
    
model = BiomassModel()
model


BiomassModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

In [5]:
criterion = WeightedMSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

class Trainer:
    def __init__(self, model, train_loader, val_loader, test_loader, criterion, optimizer, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.model.to(self.device)

    def train_epoch(self):
        self.model.train()
        running_loss = 0.0
        for images, targets in self.train_loader:
            images, targets = images.to(self.device), targets.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            running_loss += loss.item() * images.size(0)
        epoch_loss = running_loss / len(self.train_loader.dataset)
        return epoch_loss

    def infer(self, loader):
        self.model.eval()
        all_outputs = []
        with torch.no_grad():
            for images, _ in loader:
                images = images.to(self.device)
                outputs = self.model(images)
                all_outputs.append(outputs.cpu())
        all_outputs = torch.cat(all_outputs)
        return all_outputs
        
    def evaluate(self, loader):
        self.model.eval()
        all_targets = []
        all_outputs = []
        with torch.no_grad():
            for images, targets in loader:
                images, targets = images.to(self.device), targets.to(self.device)
                outputs = self.model(images)
                all_targets.append(targets.cpu())
                all_outputs.append(outputs.cpu())
        all_targets = torch.cat(all_targets)
        all_outputs = torch.cat(all_outputs)
        score = weighted_r2_score(all_targets.numpy(), all_outputs.numpy())
        return score
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
trainer = Trainer(model, train_loader, val_loader, test_loader, criterion, optimizer, device)
num_epochs = 30
for epoch in range(num_epochs):
    train_loss = trainer.train_epoch()
    val_score = trainer.evaluate(trainer.val_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Weighted R2 Score: {val_score:.4f}")

Using device: cuda
Epoch 1/30, Train Loss: 224.0566, Val Weighted R2 Score: -1.0099
Epoch 2/30, Train Loss: 66.8917, Val Weighted R2 Score: -3.1193
Epoch 3/30, Train Loss: 34.6978, Val Weighted R2 Score: -2.0586
Epoch 4/30, Train Loss: 31.0178, Val Weighted R2 Score: -1.1913
Epoch 5/30, Train Loss: 24.7396, Val Weighted R2 Score: -0.2258
Epoch 6/30, Train Loss: 19.7935, Val Weighted R2 Score: -0.0267
Epoch 7/30, Train Loss: 17.0958, Val Weighted R2 Score: 0.1098
Epoch 8/30, Train Loss: 15.6809, Val Weighted R2 Score: 0.0995
Epoch 9/30, Train Loss: 15.9499, Val Weighted R2 Score: 0.0904
Epoch 10/30, Train Loss: 13.5063, Val Weighted R2 Score: -0.0534
Epoch 11/30, Train Loss: 13.1075, Val Weighted R2 Score: 0.0533
Epoch 12/30, Train Loss: 12.7629, Val Weighted R2 Score: -0.2378
Epoch 13/30, Train Loss: 9.3086, Val Weighted R2 Score: 0.2393
Epoch 14/30, Train Loss: 6.2940, Val Weighted R2 Score: -0.1867
Epoch 15/30, Train Loss: 5.5309, Val Weighted R2 Score: 0.0788
Epoch 16/30, Train Loss

In [6]:
trainer.infer(test_loader)

tensor([[27.9683, 32.5087,  0.0000, 28.2828, 60.9322]])

In [7]:
torch.save(trainer.model.state_dict(), 'biomass_model.pth')